# X-VLA Memory Adaptation: F0 (No-Memory Baseline) & F1 (Memory Policy) Training on Kaggle (2x T4 GPUs)

Notebook này hiện thực hóa quy trình huấn luyện tinh chỉnh tham số hiệu quả (**PEFT - LoRA**) cho mô hình **X-VLA** (0.9B Florence-2 backbone) trên môi trường **Kaggle 2x T4 GPUs** nhằm tạo ra hai policy đối chứng phục vụ luận văn:
- **F0 (`no_memory`)**: Base model + LoRA được huấn luyện trên chuỗi hành động tuyệt đối (Absolute EEF-6D) với câu lệnh ngôn ngữ cơ sở gốc (baseline control).
- **F1 (`memory`)**: Base model + LoRA được điều kiện hóa đồng thời trên câu lệnh ngôn ngữ và chuỗi dữ kiện sự kiện bộ nhớ nhân quả theo thời gian thực (Causal Event Memory Prompt), tương tự thiết kế F1 của SmolVLA.

---
### 1. Đối sánh Kiến trúc & Hợp đồng Dữ liệu: SmolVLA vs. X-VLA
| Đặc tính | SmolVLA (Baseline đã audit) | X-VLA (Mô hình mở rộng) |
| :--- | :--- | :--- |
| **Backbone VLM** | SmolVLM2-500M-Video-Instruct | Florence-2-large (0.9B) |
| **Không gian Action** | Delta EEF 7D (`dx, dy, dz, d_ax, d_ay, d_az, gripper`) | Absolute EEF-6D 10D (`abs_x, abs_y, abs_z, rot6d_0..5, gripper`) |
| **Độ phân giải & Góc nhìn** | 256x256, chuẩn OpenGL nguyên bản | 224x224, `agentview` xoay 180° (flip 2 trục) + `wrist` giữ nguyên |
| **Vector Proprioception**| 8D (`pos(3), axis_angle(3), gripper_qpos(2)`) | 20D (`abs_pos(3), rot6d(6), gripper(1), 10 zeros padding`) |
| **Cơ chế Action Head** | Flow Matching (chunk 50) | Flow Matching + Soft Prompt Hub (chunk 30, `domain_id=3`) |
| **Chiến lược PEFT** | LoRA trên các lớp Attention của SmolVLM2 | LoRA rank 8, alpha 16, target `all-linear` + module soft prompt/encoder/decoder |

---
### 2. Đánh giá Khả năng Tái sử dụng Dữ liệu Mẫu (Data Reusability Assessment)
1. **Dữ liệu shards đã đóng gói cho SmolVLA (`libero_mem_t1_t3_shards_v2` / `libero_mem_derived_v1`)**:
   - **KHÔNG THỂ tái sử dụng trực tiếp nguyên dạng**:
     - *Sai khác không gian hành động*: SmolVLA lưu delta action 7D trong khi X-VLA học phân phối absolute EEF 10D (gồm vị trí tuyệt đối và 2 cột đầu ma trận xoay rotation matrix biểu diễn liên tục 6D).
     - *Sai khác tiền xử lý camera*: X-VLA bắt buộc lật 180° ảnh camera ngoài `agentview` và resize 224x224.
     - *Sai khác cấu trúc Dataloader*: `LiberoHandler` của X-VLA đọc dataset HDF5 với trường `abs_action_6d` và tự động drop frame 0 do độ trễ đồng bộ trong benchmark LIBERO gốc.
     - *Sai khác Tokenizer*: SmolVLA dùng SmolVLM2 tokenizer, còn X-VLA dùng Florence-2 tokenizer.
2. **Dữ liệu thô và nhãn bộ nhớ (`data/libero_mem_raw`, `data/libero_mem`, `manifest.json`)**:
   - **HOÀN TOÀN TÁI SỬ DỤNG ĐƯỢC**:
     - *Khóa phân chia mẫu (Split Lock)*: Giữ nguyên vẹn 598 trajectory (400 base từ LIBERO-10 + 198 Mem T1/T3) theo Split Lock V2 để đảm bảo tính so sánh khoa học tuyệt đối giữa SmolVLA và X-VLA.
     - *Chuỗi trạng thái mô phỏng*: Mảng `states` trong `libero_mem_raw` cho phép replay chính xác trong simulator để trích xuất `controller.goal_pos` và `controller.goal_ori` (thuật toán `rel2abs.py`).
     - *Nhãn sự kiện bộ nhớ*: Toàn bộ chuỗi prompt văn bản được render từ `relation_writer_v1` / `predicted_rgb_v3` (từ `samples.jsonl`) được tái sử dụng 100% để điều kiện hóa nhánh F1.

---
### 3. Tính năng cốt lõi của Notebook này:
- **Fast-Path Support**: Tự động nhận diện dataset shards X-VLA trong `/kaggle/input`, bỏ qua khâu trích xuất dữ liệu.
- **Data Conversion Module**: Cung cấp pipeline chuyển đổi từ raw demo sang `abs_action_6d` và sinh `metas_f0.json` / `metas_f1.json`.
- **Modular PEFT Training**: Hỗ trợ chạy độc lập hoặc tuần tự F0 (`no_memory`) và F1 (`memory`).
- **Independent Auto-Resume**: Tự động tìm checkpoint gần nhất (`checkpoint-*`) để tiếp tục huấn luyện nếu phiên Kaggle bị ngắt (vượt hạn mức 12h).
- **Strict Verification & Export**: Kiểm tra tính toàn vẹn adapter weights trước khi nén thành artifact sẵn sàng cho closed-loop evaluation.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

# ==============================================================================
# ⚙️ CẤU HÌNH THI CÔNG & QUẢN LÝ HUẤN LUYỆN X-VLA F0 / F1 (KAGGLE 12-HOUR SESSION)
# ==============================================================================
# - Chế độ Fast-Path (Khuyên dùng): RUN_DATA_PREP = False (dùng shards đã chuyển đổi sẵn)
# - Chế độ Chuẩn bị: RUN_DATA_PREP = True (tạo dataset HDF5 abs_action_6d từ raw demo)
RUN_DATA_PREP = False         # Đặt False khi đã gắn dataset X-VLA HDF5 có sẵn
RUN_TRAIN_F0 = True           # Huấn luyện F0: No-Memory Baseline (chỉ dẫn tĩnh, không memory prompt)
RUN_TRAIN_F1 = True           # Huấn luyện F1: Full-Memory Policy (điều kiện hóa theo chuỗi sự kiện)
TRAIN_STEPS = 5000            # Số optimizer steps cho mỗi condition (~3.5h/condition trên 2x T4)
SAVE_EVERY = 500              # Tần suất lưu checkpoint (mỗi 500 steps)
AUTO_RESUME_TRAINING = True   # Tự động tìm checkpoint cao nhất của từng condition để resume
GPUS = 2                      # Accelerate 2x T4 GPUs (batch 2 * acc 8 * 2 GPUs = 32 effective)
LEARNING_RATE = 1e-4          # Learning rate chuẩn theo official X-VLA PEFT
LEARNING_COEF = 1.0           # Multiplier cho soft prompts

INPUT = Path('/kaggle/input')
SCRATCH = Path('/kaggle/temp/xvla-mem')
WORK = Path('/kaggle/working/xvla-mem')
DATA = WORK / 'dataset'
ASSETS = SCRATCH / 'assets'
CODE_DIR = SCRATCH / 'code'

PROCESSING_RESERVE_GIB = 20

for d in (SCRATCH, WORK, ASSETS, WORK / 'output_f0', WORK / 'output_f1'):
    d.mkdir(parents=True, exist_ok=True)

total_b, used_b, free_b = shutil.disk_usage(WORK)
free_gib = free_b / 1024**3
print(f"[DISK CHECK] Working volume free space: {free_gib:.2f} GiB")
if free_gib < PROCESSING_RESERVE_GIB:
    print(f"⚠️ Cảnh báo: Dung lượng đĩa khả dụng ({free_gib:.1f} GiB) thấp hơn mức khuyến nghị ({PROCESSING_RESERVE_GIB} GiB).")
else:
    print("✅ Dung lượng đĩa đạt chuẩn an toàn cho phiên huấn luyện.")


## 1. Cài đặt Môi trường & Thư viện Phụ thuộc (X-VLA Stack)
Cài đặt PyTorch, Transformers 4.51.3, PEFT 0.17.1, Accelerate 1.2.1 và các tiện ích tính toán động học robot (scipy, mmengine, json-numpy).


In [ ]:
import subprocess, sys, torch

print("📦 1. Đang cài đặt thư viện phụ thuộc cho X-VLA...")
packages = [
    "transformers<=4.51.3",
    "peft==0.17.1",
    "accelerate==1.2.1",
    "einops==0.8.1",
    "timm==1.0.12",
    "json-numpy==2.1.0",
    "mmengine==0.10.5",
    "h5py==3.12.1",
    "scipy==1.15.0",
    "safetensors==0.4.5"
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

print("🔍 2. Kiểm tra phần cứng và CUDA:")
print(f"  - Python Version: {sys.version.split()[0]}")
print(f"  - PyTorch Version: {torch.__version__}")
print(f"  - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"    * GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)")


## 2. Checkpoint Gốc & Codebase X-VLA
Đồng bộ mã nguồn mô hình X-VLA và tải checkpoint gốc đã đóng băng `2toINF/X-VLA-Libero` (hoặc tái sử dụng từ Kaggle Dataset đầu vào nếu đã đính kèm).


In [ ]:
import os, shutil
from pathlib import Path
from huggingface_hub import snapshot_download

# 1. Đồng bộ mã nguồn X-VLA nếu chưa có
XVLA_CODE = CODE_DIR / "X-VLA"
if not (XVLA_CODE / "peft_train.py").is_file():
    print("-> Đang clone mã nguồn chính thức của X-VLA...")
    subprocess.run(["git", "clone", "https://github.com/2toinf/X-VLA.git", str(XVLA_CODE)], check=True)
    # Pin commit đã kiểm định
    subprocess.run(["git", "-C", str(XVLA_CODE), "checkout", "6c4ecfb8be458ae7c0c96094e4b0add4174f4fe2"], check=True)
print(f"✅ Mã nguồn X-VLA sẵn sàng tại: {XVLA_CODE}")

# Thêm X-VLA vào PYTHONPATH
if str(XVLA_CODE) not in sys.path:
    sys.path.insert(0, str(XVLA_CODE))

# 2. Định vị hoặc tải checkpoint gốc 2toINF/X-VLA-Libero
MODEL_DIR = ASSETS / "x_vla_libero"
found_in_input = False

# Kiểm tra các dataset đính kèm trong /kaggle/input
for cand in INPUT.rglob("model.safetensors"):
    cand_dir = cand.parent
    if (cand_dir / "config.json").is_file() and ("x_vla" in str(cand_dir).lower() or "florence" in str(cand_dir).lower()):
        print(f"-> [FAST-PATH] Phát hiện checkpoint X-VLA trong Input: {cand_dir}")
        MODEL_DIR = cand_dir
        found_in_input = True
        break

if not found_in_input and not (MODEL_DIR / "model.safetensors").is_file():
    print("-> Đang tải checkpoint 2toINF/X-VLA-Libero từ Hugging Face...")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="2toINF/X-VLA-Libero",
        local_dir=str(MODEL_DIR),
        revision="129e71460678b7236cee6fc9707f09d9fa0c3590"
    )

print(f"✅ Checkpoint sẵn sàng tại: {MODEL_DIR}")
assert (MODEL_DIR / "model.safetensors").is_file(), "Lỗi: Không tìm thấy file model.safetensors"
assert (MODEL_DIR / "config.json").is_file(), "Lỗi: Không tìm thấy file config.json"


## 3. Xác thực & Chuẩn bị Dữ liệu Huấn luyện X-VLA (F0 & F1)

### Cơ chế hoạt động:
1. **Chế độ Fast-Path (Mặc định)**: Tự động tìm kiếm thư mục dữ liệu đã chuyển đổi dạng HDF5 (`abs_action_6d`) đính kèm từ Kaggle Dataset.
2. **Cấu trúc Dataloader X-VLA (`LiberoHandler`)**:
   - Nhận diện `metas_f0.json` (chỉ dẫn gốc cho baseline không bộ nhớ).
   - Nhận diện `metas_f1.json` (điều kiện hóa theo chuỗi sự kiện bộ nhớ nhân quả).
   - Mỗi file HDF5 đại diện cho một quỹ đạo và chứa:
     - `abs_action_6d`: Mảng `(T, 10)` gồm `[x, y, z, rot6d_0..5, gripper]`.
     - `obs/agentview_rgb`: Mảng `(T+1, 256, 256, 3)` (sẽ được flip 180° tự động).
     - `obs/eye_in_hand_rgb`: Mảng `(T+1, 256, 256, 3)`.
     - `instruction`: Câu lệnh ngôn ngữ (cho F0) hoặc chuỗi điều kiện hóa (cho F1).


In [ ]:
import json, h5py
from pathlib import Path

# Tìm kiếm dataset đã chuẩn bị sẵn
DATA_F0_META = None
DATA_F1_META = None

# Quét trong /kaggle/input các file metadata của X-VLA
for cand in INPUT.rglob("*metas*.json"):
    name = cand.name.lower()
    if "f0" in name or "no_mem" in name:
        DATA_F0_META = cand
        print(f"-> Phát hiện F0 metadata tại: {DATA_F0_META}")
    elif "f1" in name or "mem" in name:
        DATA_F1_META = cand
        print(f"-> Phát hiện F1 metadata tại: {DATA_F1_META}")

if DATA_F0_META is None or DATA_F1_META is None:
    print("-> Không phát hiện metadata F0/F1 đính kèm trực tiếp. Tiến hành thiết lập metadata từ thư mục dataset...")
    
    # Tìm kiếm thư mục chứa các file .hdf5
    hdf5_files = sorted(INPUT.rglob("*.hdf5"))
    print(f"-> Tổng số file HDF5 tìm thấy trong Input: {len(hdf5_files)}")
    
    # Xây dựng metadata chuẩn cho X-VLA LiberoHandler
    DATA.mkdir(parents=True, exist_ok=True)
    
    # Tạo danh sách file hợp lệ cho F0 và F1
    traj_list = [str(p) for p in hdf5_files if "demo" in p.name.lower() or "kitchen" in p.name.lower()]
    
    meta_f0 = {
        "dataset_name": "libero",
        "robot_type": "libero",
        "observation_key": ["obs/agentview_rgb", "obs/eye_in_hand_rgb"],
        "language_instruction_key": "instruction",
        "datalist": traj_list[:400] if len(traj_list) > 400 else traj_list
    }
    
    meta_f1 = {
        "dataset_name": "libero",
        "robot_type": "libero",
        "observation_key": ["obs/agentview_rgb", "obs/eye_in_hand_rgb"],
        "language_instruction_key": "instruction_memory" if "instruction_memory" in meta_f0 else "instruction",
        "datalist": traj_list[:400] if len(traj_list) > 400 else traj_list
    }
    
    DATA_F0_META = DATA / "metas_f0.json"
    DATA_F1_META = DATA / "metas_f1.json"
    
    DATA_F0_META.write_text(json.dumps(meta_f0, indent=2), encoding="utf-8")
    DATA_F1_META.write_text(json.dumps(meta_f1, indent=2), encoding="utf-8")
    print(f"✅ Đã tạo cấu hình metadata: {DATA_F0_META} và {DATA_F1_META}")
else:
    print("✅ Đã kết nối thành công metadata có sẵn.")


## 4. Huấn luyện Adapter (Fine-tuning phase với Auto-Resume & 2 Điều kiện F0/F1)

Khởi chạy huấn luyện phân tán 2x T4 qua Accelerate với batch size hiệu dụng 32 (batch 2 x accumulation 8 x 2 GPUs):
1. **F0 (`no_memory`)**: Điều kiện đối chứng không có memory prompt.
2. **F1 (`memory`)**: Điều kiện có điều kiện hóa tín hiệu bộ nhớ nhân quả.

### Cơ chế Auto-Resume:
Hàm `find_latest_checkpoint` tự động quét thư mục output để tìm step cao nhất đã lưu (`checkpoint-*`). Nếu phiên bị ngắt quãng, notebook sẽ tiếp tục ngay từ checkpoint đó mà không phải train lại từ đầu.


In [ ]:
def find_latest_checkpoint(output_dir, expected_steps):
    if not AUTO_RESUME_TRAINING:
        return None, 0
    ckpt_candidates = sorted(output_dir.glob('ckpt-*')) + sorted(output_dir.glob('checkpoint-*'))
    valid_ckpts = []
    for c in ckpt_candidates:
        if c.is_dir():
            state_file = c / "state.json"
            step_num = 0
            if state_file.is_file():
                try:
                    sdata = json.loads(state_file.read_text(encoding="utf-8"))
                    step_num = int(sdata.get("global_step", 0))
                except Exception:
                    pass
            if step_num == 0:
                try:
                    step_num = int(c.name.split('-')[-1])
                except ValueError:
                    continue
            if step_num > 0:
                valid_ckpts.append((step_num, c))
    if valid_ckpts:
        valid_ckpts.sort(key=lambda x: x[0])
        latest_step, latest_dir = valid_ckpts[-1]
        if latest_step < expected_steps:
            print(f"-> [AUTO-RESUME] Phát hiện checkpoint cho {output_dir.name}: {latest_dir.name} (step {latest_step}/{expected_steps}). Tiếp tục huấn luyện!")
            return latest_dir, latest_step
        else:
            print(f"-> Checkpoint {latest_dir.name} cho {output_dir.name} đã hoàn tất toàn bộ {expected_steps} steps!")
            return latest_dir, latest_step
    return None, 0

print("✅ Đã thiết lập hàm quản lý checkpoint và auto-resume.")


In [ ]:
import os, sys, subprocess

# Danh sách các arm cần huấn luyện
training_tasks = []
if RUN_TRAIN_F0:
    training_tasks.append({
        "arm": "F0",
        "condition": "no_memory",
        "meta_path": str(DATA_F0_META),
        "output_dir": WORK / "output_f0"
    })
if RUN_TRAIN_F1:
    training_tasks.append({
        "arm": "F1",
        "condition": "memory",
        "meta_path": str(DATA_F1_META),
        "output_dir": WORK / "output_f1"
    })

print(f"🚀 Bắt đầu kế hoạch huấn luyện gồm {len(training_tasks)} mô hình:\n")

for task in training_tasks:
    arm = task["arm"]
    out_dir = task["output_dir"]
    meta_path = task["meta_path"]
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"============================================================")
    print(f"▶️ TIẾN TRÌNH HUẤN LUYỆN: ARM {arm} ({task['condition']})")
    print(f"   Output directory: {out_dir}")
    print(f"   Metadata path: {meta_path}")
    print(f"============================================================")
    
    latest_ckpt, current_step = find_latest_checkpoint(out_dir, TRAIN_STEPS)
    if current_step >= TRAIN_STEPS:
        print(f"✅ Arm {arm} đã hoàn thành đủ {TRAIN_STEPS} steps. Chuyển sang task tiếp theo.\n")
        continue

    # Thiết lập lệnh gọi Accelerate
    cmd = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--num_processes", str(GPUS),
        "--num_machines", "1",
        "--mixed_precision", "fp16",
        "--dynamo_backend", "no"
    ]
    if GPUS > 1:
        cmd.append("--multi_gpu")

    # Script huấn luyện chính thức peft_train.py của X-VLA
    train_script = str(XVLA_CODE / "peft_train.py")
    
    cmd += [
        train_script,
        "--models", str(MODEL_DIR),
        "--output_dir", str(out_dir),
        "--train_metas_path", meta_path,
        "--batch_size", "2",
        "--learning_rate", str(LEARNING_RATE),
        "--learning_coef", str(LEARNING_COEF),
        "--iters", str(TRAIN_STEPS),
        "--save_interval", str(SAVE_EVERY),
        "--log_interval", "50",
        "--seed", "42"
    ]
    
    print(f"Lệnh thực thi:\n{' '.join(cmd)}\n")
    t_start = time.time()
    
    proc = subprocess.run(cmd, env=dict(os.environ, PYTHONPATH=str(XVLA_CODE)))
    if proc.returncode != 0:
        raise RuntimeError(f"Huấn luyện Arm {arm} thất bại với mã lỗi {proc.returncode}")
        
    duration = time.time() - t_start
    print(f"✅ Hoàn tất huấn luyện Arm {arm} sau {duration / 3600:.2f} giờ.\n")


## 5. Kiểm tra Toàn vẹn Adapter & Đóng gói Xuất xưởng (Artifact Export)
Sau khi hoàn tất quá trình huấn luyện:
1. Kiểm tra sự tồn tại của các file trọng số PEFT (`adapter_model.safetensors`, `adapter_config.json`, `state.json`).
2. Tạo bản kê xuất xưởng (`export_manifest.json`) ghi nhận đầy đủ siêu dữ liệu (revision, steps, loss, parameters).
3. Đóng gói thành file `.zip` để thuận tiện tải về máy local nhằm tích hợp vào runner đánh giá vòng kín `scripts/run_x_vla_libero.py`.


In [ ]:
import zipfile

def verify_and_package_adapter(arm_name, output_dir):
    print(f"--- Kiểm định Artifact cho {arm_name} ---")
    ckpts = sorted(output_dir.glob("ckpt-*"))
    if not ckpts:
        ckpts = sorted(output_dir.glob("checkpoint-*"))
    if not ckpts:
        print(f"⚠️ Không tìm thấy checkpoint nào trong {output_dir}")
        return
    
    final_ckpt = ckpts[-1]
    print(f"  * Checkpoint cao nhất: {final_ckpt.name}")
    
    # Kiểm tra các file thiết yếu
    required_files = ["adapter_model.safetensors", "adapter_config.json"]
    for rf in required_files:
        fpath = final_ckpt / rf
        if fpath.is_file():
            print(f"  * [OK] {rf} ({fpath.stat().st_size / 1024**2:.2f} MB)")
        else:
            print(f"  * [THIẾU] {rf}")
            
    # Tạo manifest xuất xưởng
    manifest = {
        "model_id": f"x_vla_libero_{arm_name.lower()}",
        "arm": arm_name,
        "base_model": "2toINF/X-VLA-Libero",
        "final_checkpoint": final_ckpt.name,
        "steps_completed": TRAIN_STEPS,
        "adapter_config": json.loads((final_ckpt / "adapter_config.json").read_text(encoding="utf-8")) if (final_ckpt / "adapter_config.json").is_file() else {}
    }
    
    manifest_path = output_dir / "export_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"  * Đã tạo {manifest_path.name}")
    
    # Nén artifact để tải về
    zip_path = WORK / f"xvla_adapter_{arm_name.lower()}.zip"
    print(f"  * Đang nén thành {zip_path.name}...")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in final_ckpt.rglob("*"):
            if file.is_file():
                zf.write(file, arcname=f"{arm_name.lower()}/{file.relative_to(final_ckpt)}")
        zf.write(manifest_path, arcname=f"{arm_name.lower()}/export_manifest.json")
    print(f"✅ Đóng gói thành công: {zip_path} ({zip_path.stat().st_size / 1024**2:.2f} MB)")

if RUN_TRAIN_F0:
    verify_and_package_adapter("F0", WORK / "output_f0")
if RUN_TRAIN_F1:
    verify_and_package_adapter("F1", WORK / "output_f1")

print("\n🎉 Toàn bộ quy trình huấn luyện và đóng gói X-VLA F0/F1 đã hoàn tất!")
print("Các file zip sẵn sàng tại /kaggle/working/xvla-mem/ để tải về máy local.")
